# SZbattlefield: play one shard on Colab

Plays **one shard** of a table of CPU-against-CPU matches (or of a record for learning), keeps the result on Google Drive, and packs it as a bundle. Stopped? Run the cells again, from "2. Mount Drive" on: what is already played is not played again. In a round that is played in several places at once (forgehammer loop/dist.mjs plan), set SHARDS and TOTAL to what the plan gives Colab; collect the Drive folder with the others (dist.mjs collect). A slow or stopped place's shard can be played again in another place.

**What to know before a long run**
- The free runtime stops after about 12 hours at most, and earlier when it is idle or when the compute is wanted elsewhere. Colab's FAQ lists "chess training" among the things the free tier limits when you have no compute units. A long run (hours) should be on a **paid** runtime (compute units). A small table (the check below, 49 matches) is fine on the free one.
- No key is needed or written here. A language model is not used by this notebook. (If you ever add one: read it from Colab's *Secrets* with `google.colab.userdata.get(...)`, never type it into a cell.)
- The result of a match depends only on the game commit, the CPU versions and the seed: the same table gives the same matches here, on GitHub Actions and on your PC (`node src/compare.mjs` says so).


## 1. Settings (the only cell to edit)

In [ ]:
# --- what to play -------------------------------------------------------------------------------------------------------
GAME_COMMIT        = "b8214a6dd3dd5b9379138e64ba1d3814209a5455"   # a full sha of Siege_Zeke_openbeta (it must be on GitHub; this one is the tag szb/b8214a6)
SZB_COMMIT         = "19e1aca9f1ec36798c80c3b032ee7e534f7caa7a"   # SZbattlefield: a full sha for a result you want to repeat exactly (or "main"); this notebook is in that repository
FORGEHAMMER_COMMIT = "42ece5bec05dac5cf64b0ec237326316b25e6a66"   # forgehammer (only for packing the bundle)
CPU_A, CPU_B       = "lookahead", "simple"
PERSONA            = ""          # a persona (a file of cpus/personas/, e.g. "control"; empty = the default evaluation): given to every CPU that looks ahead (lookahead, deep) named below, also as the SHADOW teacher
DECKS              = ""          # the decks: start deck ids and/or names of decks/<name>.json, each optionally @user (start-red@bigboss), a comma list; empty = every start deck of the game
STYLE              = True        # count how each seat played (attacks at the user, trades, hand/registry/board at the end of its turns, memory left): see node src/style-report.mjs <folder>
SEEDS              = "1,"        # a LIST of seeds ("1," is the single seed 1; "1,2,3"). A bare number would mean 1..N.
SIM_PRIOR          = "real"      # where a CPU that looks ahead (lookahead, deep, a reader) draws the other side's hidden cards from: "real" = its decklist (as before), "seen" = only what the seat has seen
SWAP               = False       # also play with the CPUs on the other seats
SHARDS             = [1]         # the shards to play in this notebook, out of TOTAL (forgehammer's loop/dist.mjs plan says which are Colab's). Short ones first: a free runtime may stop.
TOTAL              = 1           # the table is cut into this many shards (the same number in every place; the same shard is the same matches everywhere)
SHADOW             = ""          # with RECORD: the teacher CPU that looks at the decisions of the other (lookahead / deep)
RECORD             = False       # True: write the teacher's decisions too (a record for learning)
CPU_FILE_ON_DRIVE  = ""          # a forged CPU (forged-<name>.mjs, made by learning) kept on Drive, copied into cpus/ (CPU_A or CPU_B names it as cpus/forged-<name>.mjs)
ROOT               = "/content/drive/MyDrive/szb/colab"   # everything goes under here, in the layout dist.mjs collect reads:  <ROOT>/<NAME>/shard-<i>-of-<TOTAL>/  and  <ROOT>/<NAME>/ore/
NAME               = "check"     # the name of the table
MAX_MATCHES        = 500         # a bigger table is not started; raise it on purpose
NODE_VERSION       = "24.19.0"   # exact: part of the environment key (results of another node are not joined)

## 2. Mount Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os
BASE = f"{ROOT}/{NAME}"
os.makedirs(BASE, exist_ok=True)
print("results go under", BASE)

## 3. Node and the three repositories (a few minutes; again after the runtime was reset)

In [ ]:
import subprocess, hashlib, urllib.request, os, pathlib, shlex

def sh(cmd, check=True, cwd=None, env=None):
    # the output is shown line by line while it runs (a long shard would show nothing until it ended)
    p = subprocess.Popen(cmd, shell=True, cwd=cwd, env=env, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    lines = []
    for line in p.stdout:
        lines.append(line)
        print(line, end="", flush=True)
    p.wait()
    if check and p.returncode != 0:
        raise SystemExit(f"failed ({p.returncode}): {cmd}")
    return subprocess.CompletedProcess(cmd, p.returncode, "".join(lines), None)

# node at the exact version, checked against the published SHASUMS256.txt
NODE_DIR = f"/content/node-v{NODE_VERSION}-linux-x64"
if not os.path.exists(NODE_DIR + "/bin/node"):
    base = f"https://nodejs.org/dist/v{NODE_VERSION}"
    name = f"node-v{NODE_VERSION}-linux-x64.tar.xz"
    sums = urllib.request.urlopen(f"{base}/SHASUMS256.txt").read().decode()
    want = next(l.split()[0] for l in sums.splitlines() if l.endswith(name))
    data = urllib.request.urlopen(f"{base}/{name}").read()
    assert hashlib.sha256(data).hexdigest() == want, "node download does not match SHASUMS256.txt"
    open("/content/" + name, "wb").write(data)
    sh(f"tar -xJf /content/{name} -C /content")
NODE = NODE_DIR + "/bin/node"
sh(f"{NODE} --version")

def clone(url, ref, path):
    if not os.path.exists(path + "/.git"):
        sh(f"git -c core.autocrlf=false clone --quiet {url} {path}")
    sh(f"git fetch --quiet origin && git checkout --quiet {shlex.quote(ref)} 2>&1 || git checkout --quiet origin/{shlex.quote(ref)}", cwd=path)
    return sh("git rev-parse HEAD", cwd=path).stdout.strip()

game_sha = clone("https://github.com/kujaku102-sketch/Siege_Zeke_openbeta", GAME_COMMIT, "/content/game")
szb_sha = clone("https://github.com/kujaku102-sketch/SZbattlefield", SZB_COMMIT, "/content/SZbattlefield")
fh_sha = clone("https://github.com/kujaku102-sketch/forgehammer", FORGEHAMMER_COMMIT, "/content/forgehammer")
assert len(GAME_COMMIT) != 40 or game_sha == GAME_COMMIT, "the game is not at the commit asked for"
if CPU_FILE_ON_DRIVE:
    import shutil
    shutil.copy(CPU_FILE_ON_DRIVE, "/content/SZbattlefield/cpus/" + os.path.basename(CPU_FILE_ON_DRIVE))
print("game", game_sha, "| SZbattlefield", szb_sha, "| forgehammer", fh_sha)

## 4. Play the shard (run this cell again to continue after a stop)

In [ ]:
workers = max(1, (os.cpu_count() or 2) - 1)
env = dict(os.environ, GAME_DIR="/content/game", GAME_COMMIT=game_sha, PATH=NODE_DIR + "/bin:" + os.environ["PATH"])
def spec(cpu):
    return f"{cpu}?persona={PERSONA}" if PERSONA and cpu in ("lookahead", "deep") else cpu

for shard in SHARDS:
    OUT = f"{BASE}/shard-{shard}-of-{TOTAL}"
    args = [f"--a {shlex.quote(spec(CPU_A))}", f"--b {shlex.quote(spec(CPU_B))}", f"--seeds {shlex.quote(SEEDS)}", f"--shard {shard}/{TOTAL}", f"--workers {workers}",
            f"--max-matches {MAX_MATCHES}", f"--out {shlex.quote(OUT)}"]
    if SWAP: args.append("--swap")
    if SIM_PRIOR == "seen": args.append("--sim-prior seen")
    if SHADOW: args.append(f"--shadow {shlex.quote(spec(SHADOW))}")
    if STYLE: args.append("--style")
    if DECKS: args.append(f"--decks {shlex.quote(DECKS)}")
    if RECORD: args.append(f"--record {shlex.quote(BASE + '/ore')}")
    print(f"=== shard {shard}/{TOTAL}")
    sh(f"{NODE} src/run.mjs " + " ".join(args), cwd="/content/SZbattlefield", env=env)

## 5. Pack the result as a bundle (hashes of every file; it refuses to pack anything that looks like a key)

In [ ]:
FORGE = f"{NODE} /content/forgehammer/packages/forge/bin/forge.mjs"
for shard in SHARDS:
    target = f"{BASE}/shard-{shard}-of-{TOTAL}"
    sh(f"{FORGE} bundle pack {shlex.quote(target)} --kind eval", env=env)
    sh(f"{FORGE} bundle verify {shlex.quote(target)}", env=env)
print("done:", BASE)

## 6. The check (optional): compare with a result made elsewhere

Put the other result's folder (`raw.jsonl`, `env.json`, `config.json`) on Drive and set `OTHER`. Every match must be identical. (Matches that only one of the two has are listed, not counted as different.)

In [ ]:
OTHER = "/content/drive/MyDrive/szb/from-local/check/shard-1-of-1"   # the first check: the same 49 matches made on a PC (empty = no check); e.g. "/content/drive/MyDrive/szb/from-actions/check/shard-1-of-1"
if OTHER:
    sh(f"{NODE} src/compare.mjs {shlex.quote(BASE + '/shard-' + str(SHARDS[0]) + '-of-' + str(TOTAL))} {shlex.quote(OTHER)}", cwd="/content/SZbattlefield", env=env)